# Piloto de máscaras nucleares

**Documento interno de exploración.** Queremos cargar una pila, entender sus
canales y comparar máscaras sobre la imagen. El scaffold no se reconstruye:
`control` y `G10` son etiquetas. Su geometría no hace falta para segmentar.

No tenemos ground truth: los contornos son **propuestas sin validar**. Podemos
revisar fallos visibles, pero no calcular precisión, Dice o IoU frente a una
referencia ni proclamar un modelo ganador. Tampoco mediremos todavía morfología.

**Recorrido:** preparación → inspección → referencia clásica 2D → redes opcionales
2D → piloto 3D → revisión y guardado opcional. Por defecto, las redes están
desactivadas: `Ejecutar todo` no descarga pesos ni lanza inferencias pesadas.

## 1. Entorno y alcance de la ejecución

Usar Python 3.11/3.12 y el kernel del entorno donde se instalen las dependencias.
Desde la carpeta del equipo:

```bash
python3 -m venv tmp/venv-mascaras
source tmp/venv-mascaras/bin/activate
python -m pip install -r notebooks/requirements-base.txt
python -m ipykernel install --user --name scaffolds-mascaras --display-name 'Scaffolds: máscaras'
```

Las instalaciones son manuales; ninguna celda ejecuta `pip`. Más abajo se indica
cómo activar Cellpose y StarDist. Los pesos, máscaras y salidas se guardan bajo
`tmp/`, ignorado por Git. Los archivos de `data/` se leen sin modificarlos.

In [ ]:
import os
import sys
import json
import re
import runpy
import hashlib
import gc
import time
from pathlib import Path
from importlib import metadata


def encontrar_proyecto(inicio=Path.cwd()):
    # Funciona si el kernel arranca en notebooks/, en el equipo o en el repo.
    for padre in (inicio, *inicio.parents):
        for candidato in (padre, padre / 'entregas/cu-05-topografía-celular',
                          padre / 'Practicas/piia2-26/entregas/cu-05-topografía-celular'):
            if (candidato / 'scripts/auditar_metadatos.py').is_file() and (candidato / 'data').is_dir():
                return candidato.resolve()
    raise FileNotFoundError('Abre el kernel dentro del proyecto y comprueba que exista data/.')


PROYECTO = encontrar_proyecto()
DATOS = PROYECTO / 'data'
TEMP = PROYECTO / 'tmp/piloto-mascaras'
TEMP.mkdir(parents=True, exist_ok=True)
# Deben fijarse antes de importar los frameworks: las cachés quedan en el equipo.
os.environ['CELLPOSE_LOCAL_MODELS_PATH'] = str(TEMP / 'modelos/cellpose')
os.environ['KERAS_HOME'] = str(TEMP / 'modelos/keras')
os.environ['MPLCONFIGDIR'] = str(TEMP / 'matplotlib')

import numpy as np
import pandas as pd
import tifffile
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from scipy import ndimage as ndi
from skimage import filters, morphology, measure, segmentation, color
from IPython.display import display

np.random.seed(0)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 10, 'axes.grid': False})
VERSIONES = {p: metadata.version(p) for p in
             ['numpy', 'scipy', 'scikit-image', 'tifffile', 'matplotlib', 'pandas']}
print('Python:', sys.version.split()[0], '| proyecto:', PROYECTO.name)
print('Versiones base:', VERSIONES)

### Configuración: empezar pequeño

`control 1` es el piloto inicial. Reducimos X/Y tomando un píxel de cada dos,
conservamos Z y actualizamos la escala; esto es una **aproximación exploratoria**,
no la resolución definitiva para medir. Para revisar detalle se puede poner
`PASO_XY = 1`. `RANGO_Z = None` conserva toda la pila; un recorte Z puede truncar
núcleos y no sirve para concluir que su volumen esté completo.

Primero ejecutar los valores actuales. Después activar **una red 2D**, revisar
tres planos y, si interesa, activar su versión 3D. No hace falta procesar las seis
pilas a la vez. Cambiar la pila o el muestreo requiere ejecutar desde esta celda.

In [ ]:
PILA = 'control 1'
PASO_XY = 2
RANGO_Z = None                 # None o (inicio, fin exclusivo), índices originales.
MAX_VOXELES_TRABAJO = 12_000_000  # Evita cargar un experimento grande por accidente.

# Rectángulos (y0, y1, x0, x1), en píxeles ORIGINALES. None propone uno por RGB.
# Revisar el recuadro que se dibuja; [] desactiva la exclusión, no es lo recomendado.
CAJAS_ANOTACION = None
PERCENTILES = (1.0, 99.8)
SIGMA_UM = 0.18
AREA_MIN_UM2 = 5.0             # Parámetros de prueba, no criterios biológicos validados.
VOLUMEN_MIN_UM3 = 5.0
PROMINENCIA_SEMILLAS_UM = 0.8

EJECUTAR_CELLPOSE_2D = False
EJECUTAR_STARDIST_2D = False
EJECUTAR_CLASICO_3D = False    # Activar tras revisar 2D; watershed 3D puede tardar.
EJECUTAR_CELLPOSE_3D = False
PERMITIR_CELLPOSE_CPU = False
MODELOS_CELLPOSE = ('cpsam_v2',)  # Alternativa moderna más pequeña: ('cpdino-vitb',).
MODELO_CELLPOSE_3D = MODELOS_CELLPOSE[0]
DIAMETRO_CELLPOSE_PX = None     # En píxeles de TRABAJO; dejar None al empezar.

VISORES_INTERACTIVOS = False   # True añade controles de revisión con ipywidgets.
GUARDAR_PROPUESTAS = False     # Guarda TIFF de etiquetas + JSON en tmp/, nunca en data/.

## 2. Inventario y canal nuclear

Reutilizamos el auditor ya existente para comprobar dimensiones, secuencia Z y
calibración de los XML Leica. **No se usa el DPI del TIFF.** Las seis carpetas
contienen 407 planos DAPI; los overlays y el segundo canal de G10 2 no son nuevas
muestras. La columna de canal se obtiene de `DyeName`, sin asumir que RGB signifique
tres marcadores biológicos.

In [ ]:
auditor = runpy.run_path(str(PROYECTO / 'scripts/auditar_metadatos.py'))
inventario = auditor['audit'](DATOS)
por_nombre = {fila['folder']: fila for fila in inventario}
resumen = []
for fila in inventario:
    for grupo, estado in fila['exports'].items():
        if estado['missing_z'] or estado['duplicate_z'] or estado['unexpected_z']:
            raise ValueError(f"Secuencia Z incorrecta: {fila['folder']} / {grupo}")
    if fila['shape_errors']:
        raise ValueError(f"TIFF y XML no coinciden: {fila['folder']}")
    resumen.append({'pila': fila['folder'], 'Z,Y,X': tuple(fila['shape_zyx']),
                    'canales': ', '.join(str(c['dye']) for c in fila['channels']),
                    'Δxy (µm)': fila['spacing_um']['X'], 'Δz (µm)': fila['spacing_um']['Z']})
display(pd.DataFrame(resumen).round(6))

if PILA not in por_nombre:
    raise KeyError(f'Pila desconocida: {PILA}. Opciones: {list(por_nombre)}')
info = por_nombre[PILA]
dapi = [c for c in info['channels'] if 'dapi' in (c['dye'] or '').lower()]
if len(dapi) != 1:
    raise ValueError('El canal nuclear no es inequívoco; revisar los metadatos antes de continuar.')
canal = int(dapi[0]['tag'])
espaciado_nativo = np.array([info['spacing_um'][a] for a in ('Z', 'Y', 'X')])
print(f'Piloto: {PILA} | canal {canal:02d}: {dapi[0]["dye"]}')

## 3. Cargar los planos sin mezclar canales ni overlays

Ordenamos Z **numéricamente**. En estas exportaciones RGB azules extraemos B como
aproximación a la señal nuclear visualizada; esto no recupera la intensidad
original del microscopio. Si en el futuro llega un TIFF monocanal, se conserva
directamente su señal. La barra y el texto aún están presentes en B.

In [ ]:
assert isinstance(PASO_XY, int) and PASO_XY >= 1
nz, ny, nx = info['shape_zyx']
inicio, fin = (0, nz) if RANGO_Z is None else RANGO_Z
assert 0 <= inicio < fin <= nz, 'RANGO_Z debe estar dentro de la pila.'
forma_prevista = (fin - inicio, len(range(0, ny, PASO_XY)), len(range(0, nx, PASO_XY)))
if np.prod(forma_prevista) > MAX_VOXELES_TRABAJO:
    raise ValueError('Experimento grande: aumentar PASO_XY o acotar RANGO_Z; revisar el límite a conciencia.')

patron = re.compile(rf'_z(\d+)_ch{canal:02d}_SV\.tif$', re.IGNORECASE)
archivos = sorted(((int(m[1]), f) for f in (DATOS / PILA).glob('*.tif')
                   if (m := patron.search(f.name))), key=lambda par: par[0])
assert [z for z, _ in archivos] == list(range(nz)), 'Faltan planos o hay índices duplicados.'
archivos = [(z, f) for z, f in archivos if inicio <= z < fin]
planos, candidatos_anotacion = [], np.zeros((ny, nx), dtype=bool)
ejemplo_rgb = None
for z, archivo in archivos:
    original = tifffile.imread(archivo)
    assert original.shape[:2] == (ny, nx)
    if original.ndim == 3 and original.shape[-1] in (3, 4):
        rgb = original[..., :3]
        # Texto/barra claros y neutros en el margen inferior: propuesta revisable.
        escala = np.iinfo(rgb.dtype).max if np.issubdtype(rgb.dtype, np.integer) else 1.0
        rgb01 = rgb.astype(np.float32) / escala
        neutro = (np.ptp(rgb01, axis=-1) < 0.05) & (rgb01.min(axis=-1) > 0.35)
        neutro[:int(0.8 * ny)] = False
        candidatos_anotacion |= neutro
        plano = rgb[..., 2]
    elif original.ndim == 2:
        plano = original
    else:
        raise ValueError(f'Formato no contemplado: {original.shape}')
    if z == (inicio + fin) // 2:
        ejemplo_rgb = original.copy()
    planos.append(plano[::PASO_XY, ::PASO_XY].astype(np.float32))
senal_sin_limpiar = np.stack(planos)
espaciado = espaciado_nativo * np.array([1, PASO_XY, PASO_XY])
indices_z_originales = np.array([z for z, _ in archivos])
print('Forma Z,Y,X:', senal_sin_limpiar.shape, '| espaciado Z,Y,X (µm):', espaciado.round(6))
if RANGO_Z is not None:
    print('AVISO: recorte Z; revisar truncamiento antes de interpretar cualquier geometría.')

### Excluir la anotación y normalizar el contraste

El detector siguiente **solo propone** el rectángulo de anotación de estas
exportaciones Leica. Revisar la figura y corregir `CAJAS_ANOTACION` si hace falta.
La zona se excluye en todos los planos; no se inventan píxeles debajo del texto.
Si toca un núcleo, su contorno quedará incompleto y habrá que señalarlo.

Calculamos percentiles sobre los píxeles válidos de toda la pila, usando el mismo
ajuste para todos los planos y métodos. La imagen original sigue intacta.

In [ ]:
if CAJAS_ANOTACION is None and candidatos_anotacion.any():
    yy, xx = np.where(candidatos_anotacion)
    cajas = [(max(0, int(yy.min()) - 4), min(ny, int(yy.max()) + 5),
              max(0, int(xx.min()) - 4), min(nx, int(xx.max()) + 5))]
    y0, y1, x0, x1 = cajas[0]
    if (y1 - y0) * (x1 - x0) > 0.12 * ny * nx:
        raise ValueError('Propuesta de anotación demasiado grande: revisar CAJAS_ANOTACION manualmente.')
else:
    cajas = [] if CAJAS_ANOTACION is None else list(CAJAS_ANOTACION)

validos_nativos = np.ones((ny, nx), dtype=bool)
for y0, y1, x0, x1 in cajas:
    assert 0 <= y0 < y1 <= ny and 0 <= x0 < x1 <= nx
    validos_nativos[y0:y1, x0:x1] = False
validos_xy = validos_nativos[::PASO_XY, ::PASO_XY]
validos = np.broadcast_to(validos_xy, senal_sin_limpiar.shape)
bajo, alto = np.percentile(senal_sin_limpiar[validos], PERCENTILES)
if alto <= bajo:
    raise ValueError('Sin rango de intensidad útil: revisar canal, pila y exclusión.')
senal = np.clip((senal_sin_limpiar - bajo) / (alto - bajo), 0, 1).astype(np.float32)
senal[~validos] = 0

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(ejemplo_rgb, cmap='gray')
axes[0].set_title('Plano recibido; recuadro = exclusión propuesta')
for y0, y1, x0, x1 in cajas:
    axes[0].add_patch(Rectangle((x0, y0), x1-x0, y1-y0, fill=False, color='orangered', lw=2))
z_central = len(senal) // 2
axes[1].imshow(senal[z_central], cmap='gray', vmin=0, vmax=1)
axes[1].set_title('Señal de trabajo: B normalizado y zona excluida')
for ax in axes:
    ax.set_xlabel('X (px del panel)'); ax.set_ylabel('Y (px del panel)')
fig.tight_layout()
plt.show()
print('Rectángulos originales (y0,y1,x0,x1):', cajas)
if not cajas:
    print('No se ha excluido ninguna zona: comprobar visualmente que no quede texto.')

## 4. Mirar XY, XZ e YZ antes de segmentar

Las vistas laterales permiten ver dónde empieza y termina la señal y si llega
al borde de la pila. Los ejes están en µm y mantienen las proporciones físicas:
Z no tiene el mismo espaciado que X/Y. El origen usado es local al recorte, no
una posición del scaffold.

In [ ]:
def panel_mascara(ax, imagen, etiquetas=None, paso=(1, 1), titulo=''):
    extension = (0, imagen.shape[1] * paso[1], imagen.shape[0] * paso[0], 0)
    vista = imagen
    if etiquetas is not None:
        # Colores = IDs de instancias; no representan confianza del modelo.
        vista = color.label2rgb(etiquetas, image=imagen, bg_label=0,
                                alpha=0.32, image_alpha=1, saturation=0)
    ax.imshow(vista, cmap='gray', vmin=0, vmax=1, extent=extension, aspect='equal')
    if etiquetas is not None and np.any(etiquetas):
        borde = segmentation.find_boundaries(etiquetas, mode='inner')
        rgba = np.zeros((*borde.shape, 4), dtype=np.float32)
        rgba[borde] = (1, 0.55, 0, 0.85)
        ax.imshow(rgba, extent=extension, aspect='equal')
    ax.set_title(titulo)


def vistas_ortogonales(volumen, etiquetas=None, z=None, y=None, x=None, titulo='Señal nuclear'):
    z = len(volumen)//2 if z is None else int(z)
    y = volumen.shape[1]//2 if y is None else int(y)
    x = volumen.shape[2]//2 if x is None else int(x)
    vistas = [('XY', volumen[z], None if etiquetas is None else etiquetas[z], espaciado[1:], 'X', 'Y'),
              ('XZ', volumen[:, y, :], None if etiquetas is None else etiquetas[:, y, :], espaciado[[0, 2]], 'X', 'Z'),
              ('YZ', volumen[:, :, x], None if etiquetas is None else etiquetas[:, :, x], espaciado[:2], 'Y', 'Z')]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    for ax, (nombre, imagen, lab, paso, horizontal, vertical) in zip(axes, vistas):
        panel_mascara(ax, imagen, lab, paso, nombre)
        ax.set_xlabel(horizontal + ' (µm)'); ax.set_ylabel(vertical + ' (µm)')
    fig.suptitle(f'{titulo} | Z original {indices_z_originales[z]}, Y={y}, X={x}')
    fig.tight_layout()
    plt.show()


vistas_ortogonales(senal)

## 5. Referencia sencilla: umbral y separación por instancias

Tu intuición azul/negro se traduce aquí en **Otsu**: separa intensidades de fondo
y señal tras un suavizado pequeño. Rellenamos huecos y quitamos objetos diminutos
con parámetros exploratorios expresados en µm. Un objeto conexo puede contener
dos núcleos pegados; **watershed** intenta separarlos usando máximos de distancia
como semillas. Ambos parten del mismo primer plano, pero sus IDs pueden diferir.

Empezamos por tres cortes reales (sin proyección máxima). **Los IDs 2D son
independientes entre planos; no constituyen una reconstrucción 3D.**

In [ ]:
def etiquetas_limpias(etiquetas, region_valida, minimo_pixeles):
    etiquetas = np.asarray(etiquetas, dtype=np.int32).copy()
    etiquetas[~region_valida] = 0
    # Se conservan las instancias del método; no se unen IDs vecinos.
    if etiquetas.any():
        etiquetas = morphology.remove_small_objects(etiquetas, min_size=max(1, int(minimo_pixeles)))
        etiquetas, _, _ = segmentation.relabel_sequential(etiquetas)
    return etiquetas.astype(np.uint32)


def segmentacion_clasica(imagen, region_valida, pasos, minimo_fisico):
    suavizada = ndi.gaussian_filter(imagen, sigma=SIGMA_UM / np.asarray(pasos))
    muestra = suavizada[region_valida]
    minimo = max(1, int(np.ceil(minimo_fisico / np.prod(pasos))))
    if not muestra.size or np.ptp(muestra) == 0:
        vacia = np.zeros(imagen.shape, dtype=np.uint32)
        return vacia, vacia.copy(), np.nan
    umbral = filters.threshold_otsu(muestra)
    binaria = (suavizada > umbral) & region_valida
    binaria = ndi.binary_fill_holes(binaria) & region_valida
    binaria = morphology.remove_small_objects(binaria, min_size=minimo)
    componentes = measure.label(binaria, connectivity=1)
    if not componentes.any():
        return componentes.astype(np.uint32), componentes.astype(np.uint32), float(umbral)
    distancia = ndi.distance_transform_edt(binaria, sampling=pasos)
    semillas = measure.label(morphology.h_maxima(distancia, PROMINENCIA_SEMILLAS_UM), connectivity=1)
    # Cada componente necesita al menos una semilla, aunque sea muy fino.
    for region in measure.regionprops(componentes):
        coords = tuple(region.coords.T)
        if not semillas[coords].any():
            punto = region.coords[np.argmax(distancia[coords])]
            semillas[tuple(punto)] = semillas.max() + 1
    separadas = segmentation.watershed(-distancia, semillas, mask=binaria)
    return (etiquetas_limpias(componentes, region_valida, minimo),
            etiquetas_limpias(separadas, region_valida, minimo), float(umbral))


z_prueba = sorted(set(int(z) for z in np.linspace(0.25*(len(senal)-1), 0.75*(len(senal)-1), 3)))
resultados_2d = {'Otsu + componentes': {}, 'Otsu + watershed': {}}
resultados_3d = {}
registro = []
for z in z_prueba:
    t0 = time.perf_counter()
    componentes, watershed, umbral = segmentacion_clasica(senal[z], validos_xy, espaciado[1:], AREA_MIN_UM2)
    resultados_2d['Otsu + componentes'][z] = componentes
    resultados_2d['Otsu + watershed'][z] = watershed
    registro.append({'prueba': 'clásica 2D', 'z_original': int(indices_z_originales[z]),
                     'segundos': round(time.perf_counter()-t0, 2), 'umbral': round(umbral, 3)})
print('Planos originales para comparar:', indices_z_originales[z_prueba].tolist())

In [ ]:
def comparar_2d(z):
    metodos = [(nombre, planos[z]) for nombre, planos in resultados_2d.items() if z in planos]
    fig, axes = plt.subplots(1, 1+len(metodos), figsize=(4*(1+len(metodos)), 3.8))
    panel_mascara(axes[0], senal[z], paso=espaciado[1:], titulo='Entrada común')
    for ax, (nombre, etiquetas) in zip(axes[1:], metodos):
        panel_mascara(ax, senal[z], etiquetas, espaciado[1:], nombre)
    for ax in axes:
        ax.set_xlabel('X (µm)'); ax.set_ylabel('Y (µm)')
    fig.suptitle(f'{PILA} | plano Z original {indices_z_originales[z]} | propuestas sin validar')
    fig.tight_layout()
    plt.show()


comparar_2d(z_prueba[len(z_prueba)//2])

## 6. Cellpose: modelos preentrenados, primero en 2D

Instalación opcional en el kernel elegido:
```python
# Ejecutar manualmente si se quiere usar esta sección:
# %pip install -r requirements-cellpose.txt
```
La referencia de API usada es **Cellpose 4.2.1.1** (`CellposeModel`). Probamos
`cpsam_v2`; `cpdino-vitb` permite otra prueba con un backbone más pequeño.
No usamos la antigua opción `model_type='nuclei'` de Cellpose 3.
Los pesos se descargan al activar por primera vez el modelo: necesitan red y disco.

Activar `EJECUTAR_CELLPOSE_2D` y ejecutar desde configuración. Cada modelo recibe
los mismos tres planos, señal y exclusión. La llamada usa canales explícitos,
un lote pequeño y libera el modelo antes de cargar el siguiente.

In [ ]:
def preparar_cellpose(nombre):
    try:
        import torch
        from cellpose import models
    except ImportError as error:
        raise ImportError('Instala notebooks/requirements-cellpose.txt en este kernel y reinícialo.') from error
    VERSIONES.update({p: metadata.version(p) for p in ('cellpose', 'torch')})
    if VERSIONES['cellpose'] != '4.2.1.1':
        print('API preparada para Cellpose 4.2.1.1; versión instalada:', VERSIONES['cellpose'])
    dispositivo = torch.device('cuda' if torch.cuda.is_available() else
                               'mps' if torch.backends.mps.is_available() else 'cpu')
    if dispositivo.type == 'cpu' and not PERMITIR_CELLPOSE_CPU:
        raise RuntimeError('Sin GPU disponible. Para una prueba CPU consciente, activar PERMITIR_CELLPOSE_CPU.')
    print(f'Cellpose {nombre} en {dispositivo}; la primera carga puede descargar pesos.')
    bf16 = dispositivo.type == 'cuda' and torch.cuda.is_bf16_supported()
    return models.CellposeModel(pretrained_model=nombre, device=dispositivo, use_bfloat16=bf16)


def inferir_cellpose(modelo, imagen, region_valida, es_3d=False):
    ceros = np.zeros_like(imagen)
    # Tres canales de entrada, pero solo uno contiene señal DAPI.
    entrada = np.stack((imagen, ceros, ceros), axis=1 if es_3d else -1)
    parametros = dict(channel_axis=1 if es_3d else -1, batch_size=1, normalize=False,
                      diameter=DIAMETRO_CELLPOSE_PX, do_3D=es_3d,
                      max_size_fraction=1.0, flow_threshold=0.4, cellprob_threshold=0.0)
    minimo = int(np.ceil((VOLUMEN_MIN_UM3 / np.prod(espaciado)) if es_3d else
                         (AREA_MIN_UM2 / np.prod(espaciado[1:]))))
    parametros['min_size'] = max(1, minimo)
    if es_3d:
        if not np.isclose(espaciado[1], espaciado[2], rtol=1e-4):
            raise ValueError('Esta llamada presupone Δx≈Δy; revisar el remuestreo para otro caso.')
        parametros.update(z_axis=0, anisotropy=float(espaciado[0]/espaciado[1]))
    etiquetas = modelo.eval(entrada, **parametros)[0]
    if etiquetas.shape != imagen.shape:
        raise ValueError(f'Salida inesperada: {etiquetas.shape} frente a {imagen.shape}')
    return etiquetas_limpias(etiquetas, region_valida, minimo)


def liberar_cellpose():
    gc.collect()
    if 'torch' in sys.modules and sys.modules['torch'].cuda.is_available():
        sys.modules['torch'].cuda.empty_cache()


if EJECUTAR_CELLPOSE_2D:
    for nombre in MODELOS_CELLPOSE:
        modelo = preparar_cellpose(nombre)
        resultados_2d['Cellpose ' + nombre] = {}
        try:
            for z in z_prueba:
                t0 = time.perf_counter()
                resultados_2d['Cellpose ' + nombre][z] = inferir_cellpose(modelo, senal[z], validos_xy)
                registro.append({'prueba': nombre + ' 2D', 'z_original': int(indices_z_originales[z]),
                                 'segundos': round(time.perf_counter()-t0, 2)})
        finally:
            del modelo
            liberar_cellpose()
else:
    print('Cellpose 2D omitido. Activar su interruptor cuando quieras descargar pesos y probarlo.')

### StarDist 2D opcional: otro candidato para núcleos fluorescentes

Usa `2D_versatile_fluo`, no el modelo RGB de histología. Su hipótesis de formas
estrelladas puede no describir todos los núcleos alargados. Esta prueba produce
instancias de cada plano por separado; no asigna identidades a través de Z.

Instalar manualmente `notebooks/requirements-stardist.txt`. Añade TensorFlow;
si se prefieren entornos separados se puede ejecutar el notebook en otro kernel.
Aquí StarDist usa CPU para no reservar GPU a la vez que PyTorch. La primera
carga también descarga pesos. Las llamadas de redes se dejan sin ejecutar en
la comprobación inicial de este documento.

In [ ]:
if EJECUTAR_STARDIST_2D:
    try:
        import tensorflow as tf
        # Fijar antes de inicializar el runtime de TensorFlow; no afecta a PyTorch.
        tf.config.set_visible_devices([], 'GPU')
        from stardist.models import StarDist2D
    except ImportError as error:
        raise ImportError('Instala notebooks/requirements-stardist.txt en el kernel y reinícialo.') from error
    except RuntimeError as error:
        raise RuntimeError('TensorFlow ya inicializó GPU: reinicia el kernel para esta prueba CPU.') from error
    VERSIONES.update({p: metadata.version(p) for p in ('stardist', 'csbdeep', 'tensorflow')})
    modelo_sd = StarDist2D.from_pretrained('2D_versatile_fluo')
    resultados_2d['StarDist fluo'] = {}
    try:
        for z in z_prueba:
            t0 = time.perf_counter()
            etiquetas, _ = modelo_sd.predict_instances(senal[z], axes='YX',
                                                        n_tiles=(1, 1), show_tile_progress=False)
            minimo = int(np.ceil(AREA_MIN_UM2 / np.prod(espaciado[1:])))
            resultados_2d['StarDist fluo'][z] = etiquetas_limpias(etiquetas, validos_xy, minimo)
            registro.append({'prueba': 'StarDist 2D', 'z_original': int(indices_z_originales[z]),
                             'segundos': round(time.perf_counter()-t0, 2)})
    finally:
        del modelo_sd
        tf.keras.backend.clear_session()
        gc.collect()
else:
    print('StarDist omitido; no se ha importado TensorFlow ni descargado ningún modelo.')

### Comparar las propuestas sobre el mismo plano

Revisar contorno exterior, núcleos unidos, fragmentación, zonas débiles y texto.
Los nucleolos pueden ser muy brillantes sin ser núcleos separados. Cambiar el
plano de la llamada o activar el visor para inspeccionar los tres cortes de prueba.

In [ ]:
comparar_2d(z_prueba[len(z_prueba)//2])
if VISORES_INTERACTIVOS:
    try:
        import ipywidgets as widgets
        display(widgets.interactive(comparar_2d,
                z=widgets.Dropdown(options=[(f'Z original {indices_z_originales[z]}', z) for z in z_prueba],
                                   value=z_prueba[len(z_prueba)//2], description='Plano:')))
    except ImportError:
        print('Instala ipywidgets o llama comparar_2d(z_prueba[0]), etc., para revisar otros planos.')

## 7. Piloto volumétrico: una etiqueta por objeto en Z

El método clásico siguiente opera en todo el volumen de trabajo y usa distancias
físicas para las semillas de watershed. Queda **desactivado por defecto** porque
puede tardar más que las comparaciones 2D. Cellpose 3D también está desactivado;
al activarlo procesa solo esta pila y corrige la anisotropía. No apilamos las
máscaras 2D como si sus IDs fueran identidades 3D.

Revisar primero los cortes 2D. Para probar watershed, activar
`EJECUTAR_CLASICO_3D`; para probar Cellpose 3D, activar `EJECUTAR_CELLPOSE_3D`
y configurar el entorno de Cellpose. Para resolución nativa, poner `PASO_XY=1`
y ajustar el límite de vóxeles conscientemente. Revisar consumo de memoria: el
tiempo depende del equipo y los pesos de Cellpose pueden tardar en descargarse.

In [ ]:
if EJECUTAR_CLASICO_3D:
    t0 = time.perf_counter()
    _, etiquetas_3d, umbral_3d = segmentacion_clasica(senal, validos, espaciado, VOLUMEN_MIN_UM3)
    resultados_3d['Clásico 3D'] = etiquetas_3d
    registro.append({'prueba': 'Clásico 3D', 'z_original': 'volumen de trabajo',
                     'segundos': round(time.perf_counter()-t0, 2), 'umbral': round(umbral_3d, 3)})

if EJECUTAR_CELLPOSE_3D:
    modelo = preparar_cellpose(MODELO_CELLPOSE_3D)
    try:
        t0 = time.perf_counter()
        nombre_3d = 'Cellpose 3D ' + MODELO_CELLPOSE_3D
        resultados_3d[nombre_3d] = inferir_cellpose(modelo, senal, validos, es_3d=True)
        registro.append({'prueba': nombre_3d, 'z_original': 'volumen de trabajo',
                         'segundos': round(time.perf_counter()-t0, 2)})
    finally:
        del modelo
        liberar_cellpose()
else:
    print('Cellpose 3D omitido: reservado para ejecución manual.')

In [ ]:
if resultados_3d:
    metodo_3d = next(iter(resultados_3d))
    vistas_ortogonales(senal, resultados_3d[metodo_3d], titulo=metodo_3d + ' — propuesta')

    def revisar_3d(metodo, z, y, x):
        vistas_ortogonales(senal, resultados_3d[metodo], z=z, y=y, x=x, titulo=metodo + ' — propuesta')

    if VISORES_INTERACTIVOS:
        try:
            import ipywidgets as widgets
            display(widgets.interactive(revisar_3d,
                metodo=widgets.Dropdown(options=list(resultados_3d), description='Método:'),
                z=widgets.IntSlider(min=0, max=senal.shape[0]-1, value=len(senal)//2, continuous_update=False),
                y=widgets.IntSlider(min=0, max=senal.shape[1]-1, value=senal.shape[1]//2, continuous_update=False),
                x=widgets.IntSlider(min=0, max=senal.shape[2]-1, value=senal.shape[2]//2, continuous_update=False)))
        except ImportError:
            print('Sin ipywidgets: usar revisar_3d(metodo_3d, z, y, x) manualmente.')
else:
    print('No hay propuestas 3D: activar al menos un método volumétrico.')

## 8. Registro mínimo para la siguiente revisión

El número de instancias y los tiempos ayudan a detectar resultados vacíos o
ejecuciones costosas; **no son métricas de exactitud**. Un ID que toca el borde
o una anotación excluida requiere revisión, no eliminación automática. Dos
métodos que coinciden también pueden compartir el mismo error.

Después de probar, anotar aquí: contornos demasiado pequeños/grandes, uniones,
fragmentaciones, comportamiento en Z y sensibilidad a los parámetros. La futura
validación necesitará máscaras manuales revisadas en ejemplos representativos.

In [ ]:
# Solo información operativa; dejamos volúmenes/ejes y estadísticas para otro notebook.
display(pd.DataFrame(registro).fillna('—'))
for nombre, etiquetas in resultados_3d.items():
    ids = np.unique(etiquetas); ids = ids[ids != 0]
    borde = np.zeros(etiquetas.shape, dtype=bool)
    borde[[0, -1], :, :] = True
    borde[:, [0, -1], :] = True
    borde[:, :, [0, -1]] = True
    junto_exclusion = ndi.binary_dilation(~validos) & validos
    revisar = np.unique(etiquetas[borde | junto_exclusion])
    revisar = revisar[revisar != 0]
    print(nombre, '| instancias propuestas:', len(ids), '| IDs que revisar por bordes/exclusión:', revisar.tolist())

# Rellenar tras mirar varios planos; no se selecciona automáticamente un ganador.
notas_revision = []
# Ejemplo de estructura (no es un resultado observado):
# notas_revision.append({'metodo': 'Clásico 3D', 'z_original': 24,
#                        'observacion': 'describir el fallo visible', 'accion': 'ajuste o corrección manual'})

### Guardar candidatos sin confundirlos con etiquetas revisadas

Si se activa `GUARDAR_PROPUESTAS`, se guardan IDs enteros y un JSON de procedencia
en `tmp/piloto-mascaras/`. Las salidas 2D son TIFF separados por plano; las 3D
son volúmenes ZYX. El JSON conserva muestreo, escala, exclusión y parámetros.
Todas las salidas se identifican como **propuestas sin validar**. El notebook
versionado se entrega sin outputs para no incorporar imágenes experimentales a Git.

In [ ]:
if GUARDAR_PROPUESTAS:
    from datetime import datetime, timezone
    ejecucion = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    carpeta_salida = TEMP / (PILA.replace(' ', '-') + '-' + ejecucion)
    carpeta_salida.mkdir(parents=True, exist_ok=False)
    nombres_salida = []
    for nombre, por_z in resultados_2d.items():
        identificador = re.sub(r'[^a-zA-Z0-9_-]+', '-', nombre).strip('-').lower()
        for z, etiquetas in por_z.items():
            archivo = f'propuesta-2d-{identificador}-z{indices_z_originales[z]:03d}.tif'
            tifffile.imwrite(carpeta_salida / archivo, etiquetas.astype(np.uint32),
                             photometric='minisblack', metadata={'axes': 'YX'})
            nombres_salida.append(archivo)
    for nombre, etiquetas in resultados_3d.items():
        identificador = re.sub(r'[^a-zA-Z0-9_-]+', '-', nombre).strip('-').lower()
        archivo = f'propuesta-3d-{identificador}.tif'
        tifffile.imwrite(carpeta_salida / archivo, etiquetas.astype(np.uint32),
                         photometric='minisblack', metadata={'axes': 'ZYX'})
        nombres_salida.append(archivo)
    xml = DATOS / info['numeric_xml']
    detalles = {'estado': 'propuestas_sin_validar', 'ground_truth_disponible': False,
                'pila': PILA, 'canal_dapi': canal, 'forma_nativa_zyx': info['shape_zyx'],
                'forma_trabajo_zyx': list(senal.shape), 'espaciado_trabajo_zyx_um': espaciado.tolist(),
                'z_originales': indices_z_originales.tolist(), 'paso_xy': PASO_XY,
                'cajas_excluidas_y0y1x0x1_px_nativos': cajas, 'percentiles': PERCENTILES,
                'limites_normalizacion': [float(bajo), float(alto)],
                'sigma_um': SIGMA_UM, 'area_min_um2': AREA_MIN_UM2, 'volumen_min_um3': VOLUMEN_MIN_UM3,
                'prominencia_semillas_um': PROMINENCIA_SEMILLAS_UM,
                'parametros_cellpose': {'diametro_px_trabajo': DIAMETRO_CELLPOSE_PX,
                    'flow_threshold': 0.4, 'cellprob_threshold': 0.0, 'batch_size': 1,
                    'normalize': False, 'max_size_fraction': 1.0},
                'archivos_entrada': [str(f.relative_to(DATOS)) for _, f in archivos],
                'xml': info['numeric_xml'], 'sha256_xml': hashlib.sha256(xml.read_bytes()).hexdigest(),
                'versiones': VERSIONES, 'registro': registro, 'notas_revision': notas_revision,
                'salidas': nombres_salida}
    (carpeta_salida / 'procedencia.json').write_text(json.dumps(detalles, ensure_ascii=False, indent=2))
    print('Propuestas guardadas en:', carpeta_salida.relative_to(PROYECTO))
else:
    print('Guardado desactivado. Los resultados solo están en memoria y en los outputs locales del notebook.')

## Próxima decisión

Revisar los tres planos 2D y varios cortes XY/XZ/YZ. Probar primero Cellpose en
2D; activar después 3D si los contornos son razonables. Repetir en una pila G10
para observar formas más alargadas, sin interpretar todavía diferencias biológicas.
La elección se apoyará en la revisión y las correcciones necesarias; el código
definitivo se extraerá a `.py` cuando decidamos el método.

**Fuentes de implementación** (consultadas el 27 de octubre de 2026):
[Cellpose: modelos](https://cellpose.readthedocs.io/en/latest/models.html),
[API](https://cellpose.readthedocs.io/en/latest/api.html),
[3D y anisotropía](https://cellpose.readthedocs.io/en/latest/do3d.html),
[caché de pesos](https://cellpose.readthedocs.io/en/latest/installation.html),
[StarDist y modelo fluorescente](https://github.com/stardist/stardist),
[CSBDeep](https://github.com/CSBDeep/CSBDeep/releases),
[watershed](https://scikit-image.org/docs/stable/auto_examples/segmentation/plot_watershed.html).
Las versiones usadas en cada ejecución quedan registradas; la documentación
`latest` puede cambiar.